# ConvTran: Positional Encoding Experiments

We study how the positional encoding affects ConvTran when the convolutional stem is unchanged. For each variant we record test accuracy, the time per training epoch and the peak CUDA memory during training. The pipeline was first checked on single datasets, and the variants are then compared on the 28 UEA datasets that completed the baseline.

The three variants build on the released implementation. The `upstream/ConvTran` submodule is used at its pinned commit and is not modified.

| Variant | Absolute encoding | Attention |
|---|---|---|
| tape_erpe | tAPE | eRPE (`Attention_Rel_Scl`) |
| sinusoidal | Sinusoidal (`AbsolutePositionalEncoding`) | Upstream dense attention, eRPE disabled |
| rope | None | Upstream dense attention with rotary encoding of Q and K, eRPE disabled |

The tAPE and eRPE variant is the released ConvTran configuration, and the sinusoidal variant uses an absolute encoding already defined in the upstream code. RoPE is not part of the upstream code; we add it in `scripts/positional_encodings.py`.

In the RoPE variant, Q and K are rotated by position after the split into heads and before the dot product. The values, the `emb_size ** -0.5` scale, the softmax and the output LayerNorm follow the upstream dense attention. With an embedding dimension of 16 and 8 heads, each head has two coordinates, so a single pair is rotated by one radian per position.

All variants share the convolutional stem, the upstream loader with its cached preprocessing, and the fixed stratified validation split (`random_state=1234`). They also use the same RAdam optimizer and learning rate, and evaluate the checkpoint with the lowest validation loss on the official test split. Synthetic checks of the variant selection and the RoPE dimensions are in `tests/test_positional_encodings.py`.

## Configuration

The notebook has three modes. The pilot trains each variant on BasicMotions for two epochs with seed 1234 to check the pipeline and the measurements. The comparison trains each variant on the dataset given by `COMPARISON_DATASET` for 100 epochs with the five baseline seeds. The benchmark applies the same protocol to every dataset that completed the baseline.

| Setting | Pilot | Comparison | Benchmark |
|---|---|---|---|
| Datasets | BasicMotions | `COMPARISON_DATASET` | 28 baseline datasets |
| Seeds | 1234 | 1234, 2345, 3456, 4567, 5678 | 1234, 2345, 3456, 4567, 5678 |
| Epochs | 2 | 100 | 100 |
| Batch size | 16 | 16 | 16 |
| Embedding dimension | 16 | 16 | 16 |
| Attention heads | 8 | 8 | 8 |

The pilot and the comparisons on BasicMotions and Handwriting are complete, and `MODE` is set to the benchmark. Training starts only when `RUN_TRAINING` is `True`; otherwise Run All reads the saved results. The configuration cell also checks the runner and the submodule commit.

In [7]:
from pathlib import Path
import math
import shlex
import subprocess
import sys

import pandas as pd

MODE = "benchmark"  # "pilot", "comparison" or "benchmark"
RUN_TRAINING = True  # Set to True, then Run All, to train the selected mode.
COMPARISON_DATASET = "Handwriting"

VARIANTS = ["tape_erpe", "sinusoidal", "rope"]
EXPECTED_SEEDS = [1234, 2345, 3456, 4567, 5678]
BATCH_SIZE = 16
EMB_SIZE = 16
NUM_HEADS = 8
MODES = {
    "pilot": {"seeds": [1234], "epochs": 2},
    "comparison": {"seeds": EXPECTED_SEEDS, "epochs": 100},
    "benchmark": {"seeds": EXPECTED_SEEDS, "epochs": 100},
}
assert MODE in MODES, f"MODE must be one of {sorted(MODES)}"
SEEDS = MODES[MODE]["seeds"]
EPOCHS = MODES[MODE]["epochs"]

ROOT = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "upstream" / "ConvTran" / "main.py").is_file()
    ),
    None,
)
assert ROOT is not None, "Open this notebook from the project repository."

RUNNER = ROOT / "scripts" / "run_positional_experiment.py"
UPSTREAM = ROOT / "upstream" / "ConvTran"
DATA_ROOT = ROOT / "Dataset" / "UEA" / "Multivariate_ts"
BASELINE_RUNS_PATH = ROOT / "results" / "baseline_runs.csv"
assert RUNNER.is_file(), f"Missing experiment runner: {RUNNER}"
assert BASELINE_RUNS_PATH.is_file(), f"Missing baseline results: {BASELINE_RUNS_PATH}"

pinned = subprocess.check_output(
    ["git", "-C", str(ROOT), "ls-tree", "HEAD", "upstream/ConvTran"], text=True
).split()[2]
checked_out = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
).strip()
upstream_changes = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "status", "--porcelain"], text=True
).strip()
assert checked_out == pinned, f"Submodule at {checked_out}, pinned {pinned}"
assert not upstream_changes, "The upstream submodule has local changes."

sys.path.insert(0, str(ROOT / "scripts"))
from run_baseline import read_result
from run_positional_experiment import FIELDS, RESULTS_PATH, read_metrics, run_directory

def run_dir(dataset, variant, seed):
    return run_directory(
        variant, dataset, seed, EPOCHS, BATCH_SIZE, EMB_SIZE, NUM_HEADS
    )

def log_complete(log):
    return read_result(log) is not None and read_metrics(log) is not None

def runner_args(dataset, variant, seed):
    return [
        "--variant", variant,
        "--dataset", dataset,
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--emb-size", str(EMB_SIZE),
        "--num-heads", str(NUM_HEADS),
    ]

def retry_command(dataset, variant, seed):
    python = Path(sys.executable)
    if python.is_relative_to(ROOT):
        python = python.relative_to(ROOT)
    return shlex.join([
        str(python), str(RUNNER.relative_to(ROOT)),
        *runner_args(dataset, variant, seed), "--retry",
    ])

def as_available(table):
    return table.astype(object).where(table.notna(), "not available")

print("Repository:", ROOT)
print("Python:", sys.executable)
print("Upstream commit (pinned, clean):", pinned)
print("Results CSV:", RESULTS_PATH.relative_to(ROOT))
print(f"Mode: {MODE} ({EPOCHS} epochs, seeds {SEEDS})")
print("RUN_TRAINING:", RUN_TRAINING)

Repository: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction
Python: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/.venv/bin/python
Upstream commit (pinned, clean): 148afb6ca549915b7e78b05e2ec3b4ba6e052341
Results CSV: results/positional_encoding_runs.csv
Mode: benchmark (100 epochs, seeds [1234, 2345, 3456, 4567, 5678])
RUN_TRAINING: True


## Datasets and run plan

Benchmark datasets are read from `results/baseline_runs.csv`. A dataset is eligible when each of the five baseline seeds has a valid result at 100 epochs and batch size 16. The cell stops if the file has duplicate rows for a dataset and seed, and checks that 28 datasets are selected.

MotorImagery and EigenWorms are not eligible because the baseline ran out of GPU memory at batch size 16. Their adapted pilots are documented in `03_baseline_training.ipynb`.

The plan contains every combination of dataset, variant and seed in the selected mode, which gives 28 × 3 × 5 = 420 runs in the benchmark.

In [8]:
baseline = pd.read_csv(BASELINE_RUNS_PATH)
baseline = baseline[(baseline["epochs"] == 100) & (baseline["batch_size"] == 16)]
duplicates = baseline[baseline.duplicated(["dataset", "seed"], keep=False)]
if not duplicates.empty:
    raise ValueError(f"Duplicate baseline rows:\n{duplicates}")

valid = baseline[
    baseline["accuracy"].between(0, 1)
    & baseline["test_loss"].map(lambda value: math.isfinite(float(value)))
]
BENCHMARK_DATASETS = [
    dataset
    for dataset, group in valid.groupby("dataset", sort=False)
    if sorted(group["seed"].astype(int)) == EXPECTED_SEEDS
]
assert len(BENCHMARK_DATASETS) == 28, f"Expected 28 datasets, found {len(BENCHMARK_DATASETS)}"
print(f"Benchmark datasets ({len(BENCHMARK_DATASETS)}):", ", ".join(BENCHMARK_DATASETS))
print("Separate pilots, not in the benchmark: MotorImagery, EigenWorms")

DATASETS = {
    "pilot": ["BasicMotions"],
    "comparison": [COMPARISON_DATASET],
    "benchmark": BENCHMARK_DATASETS,
}[MODE]
for dataset in DATASETS:
    for split in ("TRAIN", "TEST"):
        path = DATA_ROOT / dataset / f"{dataset}_{split}.ts"
        assert path.is_file(), f"Missing dataset file: {path}"

plan = [
    (dataset, variant, seed)
    for dataset in DATASETS
    for seed in SEEDS
    for variant in VARIANTS
]
assert len(set(plan)) == len(plan)
if MODE == "benchmark":
    assert len(plan) == 420, f"Expected 420 benchmark runs, found {len(plan)}"
print(
    f"\n{MODE}: {len(plan)} runs planned ({len(DATASETS)} datasets x "
    f"{len(VARIANTS)} variants x {len(SEEDS)} seeds, {EPOCHS} epochs)"
)

Benchmark datasets (28): BasicMotions, Epilepsy, ArticularyWordRecognition, AtrialFibrillation, CharacterTrajectories, Cricket, DuckDuckGeese, ERing, EthanolConcentration, FaceDetection, FingerMovements, HandMovementDirection, Handwriting, Heartbeat, InsectWingbeat, JapaneseVowels, Libras, LSST, NATOPS, PenDigits, PEMS-SF, RacketSports, SelfRegulationSCP1, SelfRegulationSCP2, PhonemeSpectra, StandWalkJump, UWaveGestureLibrary, SpokenArabicDigits
Separate pilots, not in the benchmark: MotorImagery, EigenWorms

benchmark: 420 runs planned (28 datasets x 3 variants x 5 seeds, 100 epochs)


## Training

With training enabled, runs are executed one at a time, and each prints its progress, dataset, variant, seed and log path. Each run calls `scripts/run_positional_experiment.py`, which starts the upstream `main.py` in a separate process with its own log. The full benchmark is long, and the cell can be stopped and run again.

Logs, checkpoints and configurations are saved under `outputs/experiments/positional_encoding/<variant>/<dataset>/seed<seed>_ep<epochs>_bs<batch>_emb<emb>_h<heads>/`. Completed runs are appended to `results/positional_encoding_runs.csv`. A run is reused only if its log for the same configuration is complete, so finished runs are not trained again.

If a run fails, the cell prints the error, the log path and the retry command, then moves on. The incomplete log is kept, and later attempts stop at that run until it is retried explicitly. The `--retry` command renames the old log to `run.failed-<UTC time>.log` before training.

In [9]:
attempts = []
if not RUN_TRAINING:
    print("Training disabled; set RUN_TRAINING = True in the configuration cell.")

for index, (dataset, variant, seed) in enumerate(plan if RUN_TRAINING else [], 1):
    log = run_dir(dataset, variant, seed) / "run.log"
    print(
        f"\n[{index}/{len(plan)}] {dataset}, {variant}, seed {seed}\n"
        f"  log: {log.relative_to(ROOT)}",
        flush=True,
    )
    process = subprocess.run(
        [sys.executable, str(RUNNER), *runner_args(dataset, variant, seed)],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    if process.stdout.strip():
        print(process.stdout.strip())
    if process.returncode:
        print(f"  FAILED with exit code {process.returncode}")
        print(process.stderr.strip())
        print(f"  Log: {log}")
        print(f"  Retry: {retry_command(dataset, variant, seed)}")
    attempts.append({
        "dataset": dataset,
        "variant": variant,
        "seed": seed,
        "exit_code": process.returncode,
    })

if attempts:
    attempts = pd.DataFrame(attempts)
    print(f"\nAttempted {len(attempts)} runs; {int((attempts['exit_code'] != 0).sum())} failed.")
    display(attempts[attempts["exit_code"] != 0])


[1/420] BasicMotions, tape_erpe, seed 1234
  log: outputs/experiments/positional_encoding/tape_erpe/basicmotions/seed1234_ep100_bs16_emb16_h8/run.log
Existing complete run: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/positional_encoding/tape_erpe/basicmotions/seed1234_ep100_bs16_emb16_h8/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/positional_encoding_runs.csv

[2/420] BasicMotions, sinusoidal, seed 1234
  log: outputs/experiments/positional_encoding/sinusoidal/basicmotions/seed1234_ep100_bs16_emb16_h8/run.log
Existing complete run: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/outputs/experiments/positional_encoding/sinusoidal/basicmotions/seed1234_ep100_bs16_emb16_h8/run.log
Saved result to: /home/tyrerodr/Documents/Msc-Advanced-AI/TimeSeries/convtran-reproduction/results/positional_encoding_runs.csv

[3/420] BasicMotions, rope, seed 1234
  log:

,dataset,variant,seed,exit_code
300,PEMS-SF,tape_erpe,1234,1
312,PEMS-SF,tape_erpe,5678,1


## Results by run

Results are read from `results/positional_encoding_runs.csv` and the run directories, so this cell also works with training disabled. Only rows matching the selected epochs, batch size, embedding dimension, number of heads, datasets, variants and seeds are used, so the two-epoch pilot appears only in its own mode.

A run is complete when the CSV has a row for its exact configuration and its log contains both the final test summary and the measurement record. A run is incomplete when its log exists but is not complete; it needs the `--retry` command. All other runs are pending. This includes a recorded row whose log is missing, which the runner retrains, and a complete log without a row, which the runner records without training.

Test accuracy and test loss come from the upstream evaluation: after training, the checkpoint with the lowest validation loss is evaluated on the official test split.

Epoch times measure each call to the upstream `train_epoch`, with CUDA synchronized at the start and end. They include batch loading, the forward and backward passes, gradient clipping and the optimizer step, but not validation, checkpoint saving, logging or the test evaluation. The first epoch is reported separately because it includes one-time costs such as CUDA and cuDNN initialization and the creation of the optimizer state. The later-epoch time is the mean over epochs 2 to the last.

Peak memory is `torch.cuda.max_memory_allocated`. The counter is reset once the model is on the GPU, at the start of the upstream `train_runner`, and read when it returns, before testing. Upstream validates after every training epoch, so the value covers training and validation. It counts tensors allocated by PyTorch, not memory reserved by the allocator, the CUDA context or other processes, so it is lower than the value reported by `nvidia-smi`. It is not available for CPU runs.

With more than 30 runs, the table shows status counts per dataset instead of individual runs.

In [10]:
results = (
    pd.read_csv(RESULTS_PATH) if RESULTS_PATH.is_file()
    else pd.DataFrame(columns=FIELDS)
)
config_fields = ["dataset", "variant", "seed"]
selected = results[
    (results["epochs"] == EPOCHS)
    & (results["batch_size"] == BATCH_SIZE)
    & (results["emb_size"] == EMB_SIZE)
    & (results["num_heads"] == NUM_HEADS)
    & results["dataset"].isin(DATASETS)
    & results["variant"].isin(VARIANTS)
    & results["seed"].isin(SEEDS)
]
if selected.duplicated(config_fields).any():
    raise ValueError("Duplicate rows for one configuration in the results CSV.")
recorded = {
    (row.dataset, row.variant, int(row.seed)): row
    for row in selected.itertuples(index=False)
}

def epoch_times(value):
    if pd.isna(value):
        return []
    return [float(part) for part in str(value).split(";") if part]

rows = []
for dataset, variant, seed in plan:
    log = run_dir(dataset, variant, seed) / "run.log"
    record = recorded.get((dataset, variant, seed))
    row = {
        "dataset": dataset, "variant": variant, "seed": seed,
        "log": str(log.relative_to(ROOT)),
    }
    if record is not None and log_complete(log):
        times = epoch_times(record.train_epoch_times_s)
        row.update(
            status="complete",
            test_accuracy_pct=record.accuracy * 100,
            test_loss=record.test_loss,
            first_epoch_s=times[0] if times else None,
            later_epochs_mean_s=(
                sum(times[1:]) / len(times[1:]) if len(times) > 1 else None
            ),
            timed_epochs=len(times),
            peak_cuda_alloc_train_val_mib=record.peak_cuda_allocated_train_val_mib,
            device=record.device,
        )
    elif log.is_file() and not log_complete(log):
        row["status"] = "incomplete"
    else:
        row["status"] = "pending"
    rows.append(row)

RUN_COLUMNS = [
    "dataset", "variant", "seed", "status", "test_accuracy_pct", "test_loss",
    "first_epoch_s", "later_epochs_mean_s", "timed_epochs",
    "peak_cuda_alloc_train_val_mib", "device", "log",
]
NUMERIC_COLUMNS = [
    "test_accuracy_pct", "test_loss", "first_epoch_s",
    "later_epochs_mean_s", "peak_cuda_alloc_train_val_mib",
]
runs_table = pd.DataFrame(rows).reindex(columns=RUN_COLUMNS)
runs_table[NUMERIC_COLUMNS] = runs_table[NUMERIC_COLUMNS].apply(pd.to_numeric)
runs_table["timed_epochs"] = runs_table["timed_epochs"].astype("Int64")

counts = runs_table["status"].value_counts()
print(f"{MODE}: {len(runs_table)} planned")
for status in ("complete", "pending", "incomplete"):
    print(f"  {status}: {counts.get(status, 0)}")

if len(runs_table) <= 30:
    display(as_available(runs_table.round(4)))
else:
    progress = (
        runs_table.pivot_table(
            index="dataset", columns="status", values="seed",
            aggfunc="count", fill_value=0,
        )
        .reindex(index=DATASETS, columns=["complete", "pending", "incomplete"], fill_value=0)
    )
    with pd.option_context("display.max_rows", None):
        display(progress)

incomplete = runs_table[runs_table["status"] == "incomplete"]
if not incomplete.empty:
    print("Incomplete logs are kept. Inspect each log, then retry from the repository root:")
    for run in incomplete.itertuples(index=False):
        print(" ", retry_command(run.dataset, run.variant, run.seed))

benchmark: 420 planned
  complete: 418
  pending: 0
  incomplete: 2


status,complete,pending,incomplete
dataset,,,
BasicMotions,15,0,0
Epilepsy,15,0,0
ArticularyWordRecognition,15,0,0
AtrialFibrillation,15,0,0
CharacterTrajectories,15,0,0
Cricket,15,0,0
DuckDuckGeese,15,0,0
ERing,15,0,0
EthanolConcentration,15,0,0


Incomplete logs are kept. Inspect each log, then retry from the repository root:
  .venv/bin/python scripts/run_positional_experiment.py --variant tape_erpe --dataset PEMS-SF --seed 1234 --epochs 100 --batch-size 16 --emb-size 16 --num-heads 8 --retry
  .venv/bin/python scripts/run_positional_experiment.py --variant tape_erpe --dataset PEMS-SF --seed 5678 --epochs 100 --batch-size 16 --emb-size 16 --num-heads 8 --retry


## Summary by dataset and variant

Completed seeds are summarized for each dataset and variant. Accuracy is the mean in percent with the sample standard deviation in percentage points. The standard deviation is not available with a single seed.

Epoch times are first averaged over epochs 2 to the last within each seed, then across seeds. First-epoch time and peak memory are averaged across seeds.

Summaries with pending seeds are labelled as partial and should not be compared directly with complete ones.

In [11]:
summary_rows = []
for dataset in DATASETS:
    for variant in VARIANTS:
        done = runs_table[
            (runs_table["dataset"] == dataset)
            & (runs_table["variant"] == variant)
            & (runs_table["status"] == "complete")
        ]
        completed = len(done)
        summary_rows.append({
            "dataset": dataset,
            "variant": variant,
            "summary": "complete" if completed == len(SEEDS) else "partial",
            "seeds_completed": f"{completed}/{len(SEEDS)}",
            "mean_accuracy_pct": done["test_accuracy_pct"].mean(),
            "std_accuracy_pp": (
                done["test_accuracy_pct"].std(ddof=1) if completed > 1 else None
            ),
            "mean_first_epoch_s": done["first_epoch_s"].mean(),
            "mean_later_epochs_s": done["later_epochs_mean_s"].mean(),
            "mean_peak_cuda_alloc_train_val_mib": (
                done["peak_cuda_alloc_train_val_mib"].mean()
            ),
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_NUMERIC = summary.columns[4:]
summary[SUMMARY_NUMERIC] = summary[SUMMARY_NUMERIC].apply(pd.to_numeric)
with pd.option_context("display.max_rows", None):
    display(as_available(summary.round(4)))

,dataset,variant,summary,seeds_completed,mean_accuracy_pct,std_accuracy_pp,mean_first_epoch_s,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib
0,BasicMotions,tape_erpe,complete,5/5,100.0,0.0,0.2214,0.0046,44.0703
1,BasicMotions,sinusoidal,complete,5/5,100.0,0.0,0.1706,0.0044,43.9683
2,BasicMotions,rope,complete,5/5,100.0,0.0,0.1933,0.0049,43.9385
3,Epilepsy,tape_erpe,complete,5/5,96.3768,0.0,0.2058,0.0178,109.0815
4,Epilepsy,sinusoidal,complete,5/5,96.2319,0.3241,0.2135,0.0168,108.9019
5,Epilepsy,rope,complete,5/5,96.6667,0.8262,0.2234,0.0181,108.8408
6,ArticularyWordRecognition,tape_erpe,complete,5/5,96.8,0.8028,0.2108,0.0319,73.5298
7,ArticularyWordRecognition,sinusoidal,complete,5/5,95.8667,0.9006,0.2046,0.0302,73.3364
8,ArticularyWordRecognition,rope,complete,5/5,95.4667,1.1205,0.2293,0.034,73.2944
9,AtrialFibrillation,tape_erpe,complete,5/5,32.0,2.9814,0.1912,0.0106,635.104


## Comparison across datasets

The comparison across datasets uses only datasets where all three variants have complete summaries, so each variant is averaged over the same datasets. The cell reports how many datasets are included.

Each dataset has equal weight, regardless of its size or difficulty. Mean accuracy is the mean of the per-dataset means; the standard deviations across seeds are not combined. Ranks are computed within each dataset from mean accuracy, with 1 for the best variant and ties sharing the average rank.

In [12]:
complete_summaries = summary[summary["summary"] == "complete"]
common = [
    dataset for dataset in DATASETS
    if set(complete_summaries.loc[complete_summaries["dataset"] == dataset, "variant"])
    == set(VARIANTS)
]
print(f"Datasets with complete results for all three variants: {len(common)} of {len(DATASETS)}")

shared = complete_summaries[complete_summaries["dataset"].isin(common)].copy()
shared["rank"] = shared.groupby("dataset")["mean_accuracy_pct"].rank(ascending=False)
overall = (
    shared.groupby("variant")
    .agg(
        datasets=("dataset", "count"),
        mean_accuracy_pct=("mean_accuracy_pct", "mean"),
        mean_rank=("rank", "mean"),
        mean_later_epochs_s=("mean_later_epochs_s", "mean"),
        mean_peak_cuda_alloc_train_val_mib=("mean_peak_cuda_alloc_train_val_mib", "mean"),
    )
    .reindex(VARIANTS)
    .reset_index()
)
overall["datasets"] = overall["datasets"].fillna(0).astype(int)
display(as_available(overall.round(4)))

Datasets with complete results for all three variants: 27 of 28


,variant,datasets,mean_accuracy_pct,mean_rank,mean_later_epochs_s,mean_peak_cuda_alloc_train_val_mib
0,tape_erpe,27,70.9825,1.7037,0.4537,1130.4464
1,sinusoidal,27,69.727,2.2778,0.4096,1126.2237
2,rope,27,69.5929,2.0185,0.4524,1126.1419


## Limitations

The five seeds of each dataset share one validation split, and each timing comes from a single run on a desktop GPU. Datasets differ in size, length and number of classes, and averages across datasets give each dataset equal weight, which hides these differences. The benchmark excludes MotorImagery and EigenWorms, which did not fit in GPU memory under the baseline configuration.

With one rotated pair per head, the RoPE angle grows by one radian per position and repeats roughly every 6.3 positions. This variant therefore encodes relative distance only coarsely, and the results apply to this configuration rather than to RoPE in general.

The two-epoch pilot only checks that the pipeline and the measurements work. It is too short to compare accuracy.